原始评论 → 文本清洗 → 分词编码 → 补长到 128 → 分批喂给模型 → 算损失 → 改参数      
例如：这款手机质量很好     
1.分词（bert-base-chinese：一字一个 token，[这, 款, 手, 机, 质, 量, 很, 好]     
2.查 vocab.txt 变成 ID（固定编号）比如这 → 6821，款 → 3633，手 → 2881     
3.强制把这句话补到 128 维（少补多裁）     
4.生成 attention_mask（告诉模型哪部分是真字，哪些是补充的）          
5.句子 1 和 句子 2 之间没有上下文、没有注意力、没有关联，模型只看每一句内部的字与字的关系，句子之间是平行独立的          
6.批次（batch）：只是为了算得快，比如设置 BATCH_SIZE=16：一次把 16 句独立句子 同时扔进 GPU，GPU 并行算 16 句的损失，最后把 16 个损失取平均，一次性改一遍模型参数    
7.模型训练，看一句话内部，哪些字组合起来是正面 / 负面，通过标签（0/1）不断修正判断，让预测越来越准    

训练过程    
1.每个字变成 Q/K/V 向量   
2.计算注意力分数：谁和谁近   
3.不断调整模型参数（Q、K、V 的三个矩阵权重和分类层权重），让注意力分数变得更合理：模型一开始是 “瞎蒙” 的,刚初始化的 BERT，注意力分数是乱的！它可能：“手” 关注 “很”，“质” 关注 “这”，“好” 关注 “款”，完全乱搭，不懂语义。训练就是：不断调整参数，让注意力分数从 “乱搭” → 慢慢变成 “合理搭配”。也就是：让相关的字（手机、质量、很好）之间分数变高，让无关的字之间分数变低，这个过程就叫：
让注意力分数越来越符合语义。     
4.标签（0/1）是 “老师”你给模型：句子+标准答案 label=1（正面），模型预测错了 → 产生 loss（损失）→ 反向传播改参数→ 让注意力分配更合理       
  看到 “质量 + 很好” → 注意力拉满 → 输出 1（正面）      
  看到 “质量 + 差” → 注意力拉满 → 输出 0（负面）

In [9]:
import re
import os
import jieba
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score,precision_score, recall_score

In [10]:
DATA_PATH ="D:/深度学习/数据集+代码/LSTM/online_shopping_10_cats.csv" 
STOPWORDS_PATH = "D:/深度学习/bert-base-chinese/stopwords.txt"  
NUM_LABELS = 2  # 分类数：二分类设2，多分类改对应数字
SAVE_MODEL_PATH = "./bert_emotion_model" 


MODEL_NAME = "D:/深度学习/bert-base-chinese"
MAX_LEN = 128  
BATCH_SIZE = 16  
LR = 2e-5 
EPOCHS = 3  
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"当前训练设备：{DEVICE}")

当前训练设备：cuda:0


<font style='font-size:30px;'>
1.中文文本预处理

In [11]:
#加载停用词表
def load_stopwords(stopword_path):
    stopwords = set()
    with open(stopword_path, "r", encoding="utf-8") as f:
        for line in f.readlines():
            word = line.strip()
            if word:
                stopwords.add(word)
    return stopwords


#文本清洗
def preprocess_text(text, stopwords):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^\u4e00-\u9fa5]", "", text)  
    word_list = jieba.lcut(text)  
    word_list = [w for w in word_list if w not in stopwords and len(w) > 1]
    if len(word_list) < 2:  
        return ""
    return " ".join(word_list)


def process_data(raw_data_path, stopwords_path):
    stopwords = load_stopwords(stopwords_path)
    df = pd.read_csv(raw_data_path, encoding="utf-8-sig")
    # 检查必要列
    assert "review" in df.columns and "label" in df.columns, "必须包含 review 和 label 列"
    
    # 批量预处理
    df["clean_text"] = df["review"].apply(lambda x: preprocess_text(x, stopwords))
    df = df[df["clean_text"] != ""].reset_index(drop=True)
    
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
    print(f"总样本：{len(df)} | 训练集：{len(train_df)} | 验证集：{len(val_df)}")
    return train_df, val_df

<font style='font-size:30px;'>
2.tokenizer文本编码

In [12]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# 构建BERT编码函数
def bert_encoder(text, tokenizer, max_len):
    encoding = tokenizer.encode_plus(
        text=text,
        add_special_tokens=True,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=False,
        return_tensors="pt"
    )
    return encoding["input_ids"].flatten(), encoding["attention_mask"].flatten()

<font style='font-size:30px;'>
3.定义Dataset

In [13]:
#构建抽象类数据集
class TextEmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df["clean_text"].tolist()   #把 pandas 的 Series 转成 Python 列表，Dataset 接收列表格式更高效
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        input_ids, attention_mask = bert_encoder(self.texts[idx], self.tokenizer, self.max_len)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": label
        }

# 构建数据加载器
def build_dataloader(train_df, val_df, tokenizer, max_len, batch_size):
    train_dataset = TextEmotionDataset(train_df, tokenizer, max_len)
    val_dataset = TextEmotionDataset(val_df, tokenizer, max_len)
    #把Dataset转换成可批量加载的DataLoader
    #（参数：数据集、每批16条、是否打乱【训练集需要打乱】、单进程加载【模型训练的时候，子进程把下一批数据准备好，节约时间防止算力浪费、Windows要求为0】）
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader

<font style='font-size:30px;'>
4.模型初始化+优化器配置

In [14]:
#构建初始化模型
def init_model(model_name, num_labels, device):
    model = BertForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,  # 这里要和全局配置的 NUM_LABELS 一致
        problem_type="single_label_classification"
    ).to(device)
    return model

#构建初始优化器
def init_optimizer(model, lr):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=0.01  #权重衰减（L2正则化）
    )
    return optimizer

<font style='font-size:30px;'>
5.训练+验证

In [15]:
# 单轮训练
def train_one_epoch(model, train_loader, optimizer, device, epoch, total_epochs):
    #训练模式，开启 Dropout训练时，模型每一次看数据都会随机屏蔽一部分神经元（看不见），逼着模型不能死记硬背，必须学真正的规律
    model.train()
    total_loss = 0.0
    #tqdm：显示训练进度条-->desc：显示当前是第几轮，比如 训练: 1/3-->enumerate：给每个批次编号-->total：总共有多少批次
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"训练: {epoch}/{total_epochs}")
    for step, batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()       #梯度清零
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)   #前向传播
        loss = outputs.loss         #获取损失
        loss.backward()             #反向传播
        optimizer.step()            #优化器更新参数

        total_loss += loss.item()   #损失累加（.item() 把张量变成普通数字，不占显存） 
        pbar.set_postfix(loss=f"{loss.item():.4f}")    #在进度条后面实时显示当前损失
    
    avg_loss = total_loss / len(train_loader)
    #准备两个空列表，存所有预测值和所有真实标签，用来算这一轮的训练准确率
    all_preds = []
    all_labels = []
    #关闭梯度计算，算准确率不需要更显参数，省现存，跑得快
    with torch.no_grad():
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            #模型给每句话打分类别分数 → 存在 logits里，例如[0.05, 0.95]，[0.92, 0.08]，[0.10, 0.90]
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            #dim=1 = 沿着 “类别维度” 取最大值的索引，最大值索引在0则预测为负类，在1则预测为正类，运行结果[1, 0, 1]
            preds = torch.argmax(outputs.logits, dim=1)
            # 把当前批次的预测结果 从GPU转到CPU → 转成普通数组只有变成数组，才能用来计算准确率、F1 分数等
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    train_acc = accuracy_score(all_labels, all_preds)
    return avg_loss, train_acc

# 单轮验证
def val_one_epoch(model, val_loader, device, epoch, total_epochs):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    
    pbar = tqdm(enumerate(val_loader), total=len(val_loader), desc=f"验证: {epoch}/{total_epochs}")
    with torch.no_grad():
        for step, batch in pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")  #进度条显示损失
            
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(val_loader)
    acc = accuracy_score(all_labels, all_preds)
    #average="binary" → 二分类专用，average="macro" → 多分类，每个类别平等算，average="weighted" → 数据不平衡时必用
    precision = precision_score(all_labels, all_preds, average="binary")
    recall = recall_score(all_labels, all_preds, average="binary")
    f1 = f1_score(all_labels, all_preds, average="binary")
    return avg_loss, acc, precision, recall, f1

In [16]:
# ================================= 主函数：端到端训练 =================================
if __name__ == "__main__":
    # 1. 数据预处理
    train_df, val_df = process_data(DATA_PATH, STOPWORDS_PATH)
    print(f"数据集大小 - 训练集：{len(train_df)}，验证集：{len(val_df)}")
    print("数据预处理完成！\n" + "="*50)
    
    # 2. 构建数据加载器
    train_loader, val_loader = build_dataloader(train_df, val_df, tokenizer, MAX_LEN, BATCH_SIZE)
    # 3. 初始化模型和优化器
    model = init_model(MODEL_NAME, NUM_LABELS, DEVICE)
    optimizer = init_optimizer(model, LR)

    # 4. 模型训练
    best_f1 = 0.0
    best_acc = 0.0
    print("开始模型训练\n" + "="*50)

    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        print("-"*50)
        # 训练
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, DEVICE, epoch+1, EPOCHS)
        # 验证
        val_loss, val_acc, val_precision, val_recall, val_f1 = val_one_epoch(model, val_loader, DEVICE, epoch+1, EPOCHS)
        
        # 打印日志
        print(f"训练集 - 损失：{train_loss:.4f}，准确率：{train_acc:.4f}")
        print(f"验证集 - 损失：{val_loss:.4f}，准确率：{val_acc:.4f}")
        print(f"精确率：{val_precision:.4f}，召回率：{val_recall:.4f}，F1分数：{val_f1:.4f}")
        
        # 保存最佳模型
        if val_acc > best_acc:
            best_acc = val_acc
            if not os.path.exists(SAVE_MODEL_PATH):
                os.makedirs(SAVE_MODEL_PATH)
            model.save_pretrained(SAVE_MODEL_PATH)
            tokenizer.save_pretrained(SAVE_MODEL_PATH)
            print(f"✅ 保存最佳模型，准确率：{val_acc:.4f}")

    print("\n" + "="*50)
    print("训练完成！")

总样本：61569 | 训练集：49255 | 验证集：12314
数据集大小 - 训练集：49255，验证集：12314
数据预处理完成！


Some weights of the model checkpoint at D:/深度学习/bert-base-chinese were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at D:/深度学习/bert-base-chinese

开始模型训练

Epoch 1/3
--------------------------------------------------


训练: 1/3:   0%|          | 2/3079 [00:09<3:52:30,  4.53s/it, loss=0.8121]


KeyboardInterrupt: 